# Surface conductance boundary condition for the 1st-order closure wind model

**Problem.** The lowest grid cells of `closure_1_model_U` produce unphysically low wind
speeds on open surfaces (snow, moss, bare fields), which propagates into the surface
energy balance of the snow model through the turbulent exchange coefficients.

**Root causes identified earlier:**
1. The mixing length near the ground contained grid-dependent constants
   ($\ell = \kappa(z + \Delta z/2) + 0.01$), giving an effective roughness length
   $z_{0,eff} \approx 0.06$ m instead of the physical
   $z_0 \sim 1$ mm for snow.
2. Even with a corrected $\ell = \kappa(z+z_0)$, a uniform grid with
   $\Delta z = 0.25$ m cannot resolve the logarithmic layer between $z_0$ and the
   first node, where half of the total velocity change occurs.

**This notebook compares three remedies** — the original model, brute-force grid
refinement, and an analytic surface-conductance boundary condition — using two solver
formulations (the original nodal scheme and a conservative finite-volume scheme), and
quantifies the momentum balance error of each.

## Recap: what `closure_1_model_U` solves

**Momentum budget** (stationary, horizontally homogeneous, neutral; from RANS):

$$\frac{d\tau}{dz} = \rho\, C_d\, \mathrm{LAD}(z)\, \overline{U}^2 + \frac{d\overline{P}}{dx}$$

The momentum flux $\tau = -\rho\,\overline{u'w'}$ is constant with height on open ground
($\mathrm{LAD} = 0$, no pressure gradient) and is depleted downward inside a canopy (drag sink).

**First-order closure** (Prandtl mixing length): the flux is tied to the local gradient,

$$\tau = \rho\, K_m \frac{d\overline{U}}{dz}, \qquad
K_m = \ell^2 \left|\frac{d\overline{U}}{dz}\right|, \qquad
\ell = \begin{cases} \kappa(z+z_0) & \text{near ground} \\ \alpha h & \text{in canopy} \\ \kappa(z-d) & \text{above canopy}\end{cases}$$

Substituting the closure into the budget gives a nonlinear 2nd-order ODE for
$\overline{U}(z)$, solved iteratively (Picard + relaxation) as a tridiagonal system.
All velocities are $u_*$-normalized: the solver works with $U/u_*$, and outputs are
scaled back by $u_*$ (velocities), $u_*$ ($K_m$, note units [m]) and $u_*^2$ ($\tau$).

**The special case.** With $\mathrm{LAD}=0$, $d\tau/dz = 0$ and $\tau = \rho u_*^2$; the closure
then integrates to the logarithmic law
$\overline{U}(z) = (u_*/\kappa)\ln(z/z_0)$ — the analytic reference used throughout
this notebook.

**MODIFY LAD** One can change the LAD to either 0, constant near groud (simulates open fields) or weibull distribution based full canopy simulations. The logarithmic law is shown together with each LAD but that is the analytical solution only for the open surface ($\mathrm{LAD}=0$ case).

## Derivation of the surface conductance boundary condition

**Grid.** Finite-volume staggering: faces on the original model grid
$z_f = 0, 0.25, \dots, 25$ m; $U$ and $\tau$-diagnostics stored at cell centres
$z_c = 0.125, \dots, 24.875$ m. The bottom face of cell 0 is the ground. We need the
momentum flux through that face, $\tau_0$, as a function of the solved unknown $U_0$.

**Assumptions** (the closure restricted to the thin layer $z_0 \le z \le z_{c,0}$):

- **(G1)** Constant flux across the layer: exact on open ground; a good approximation
  under a canopy because the drag sink $C_d a U^2$ is small where $U$ is small.
- **(G2)** Mixing-length closure with $\ell = \kappa z$ (near-ground branch).
- **(G3)** No-slip at the aerodynamic surface: $U(z_0) = 0$. The no-slip condition
  lives *inside this derivation*, not at a grid node.
- **(G4)** Neutral stratification (consistent with the rest of the model).

**Derivation.** G1 + G2 inverted for the gradient and integrated from $z_0$ to the
lowest cell centre $z_{c,0}$:

$$\frac{dU}{dz} = \frac{\sqrt{\tau_0/\rho}}{\kappa z}
\quad\Longrightarrow\quad
U_0 = \frac{\sqrt{\tau_0/\rho}}{\kappa}\ln\frac{z_{c,0}}{z_0}
\quad\Longrightarrow\quad
\boxed{\tau_0 = \rho\, g_m\, U_0^2, \qquad
g_m = \left(\frac{\kappa}{\ln(z_{c,0}/z_0)}\right)^{2}}$$

$g_m$ is the bulk transfer coefficient of the unresolved log layer — the analytic
integral of the resistance $\int_{z_0}^{z_{c,0}} dz/K_m$, done once with pen and paper
so the grid never has to resolve it.

**Implementation.** In the budget of cell 0, $\tau_0$ enters as a sink linearized in
Picard fashion ($\tau_0 = g_m U_0^{(n)} U_0^{(n+1)}$), structurally identical to the
canopy drag term. Limits: $g_m \to \infty$ recovers Dirichlet no-slip at the node;
$g_m \to 0$ recovers the zero-flux BC — both existing branches of the code are special
cases.

**Scalars for free.** The same construction with the scalar diffusivity gives the
surface conductance for $CO_2$, $H_2O$ and heat. **Mind the pyAPES convention:**
the code uses `Ks = Km * Sc` with `Sc = 2.0` in every parameter file, i.e. the
parameter is the *inverse* turbulent Schmidt number ($Sc_t = 0.5$, scalars mix
twice as efficiently as momentum inside the canopy). Consistently with the code,

$$g_s = g_m \cdot Sc_{code}, \qquad F = \rho_{mol}\, g_s\, U_0\,(C_{surface} - C_0)$$

This is the conductance the snow/soil surface schemes should receive. Writing
$g_s = g_m/Sc$ instead would be wrong by a factor of four with the present
parameter values.

## Conservative (finite-volume) formulation

The original scheme discretizes the *expanded* form
$K_m U'' + K_m' U'$ at nodes; no shared face flux exists between adjacent equations,
so momentum is numerically created where $K_m$ curves sharply (near the surface).

The FVM scheme discretizes the *flux* form. Cell $i$ budget:

$$\frac{\tau_{i+1/2} - \tau_{i-1/2}}{\Delta z} = C_d a_i U_i^2 - \frac{dP}{dx},
\qquad \tau_{i+1/2} = K_{i+1/2}\,\frac{U_{i+1}-U_i}{\Delta z}$$

with $K_{i+1/2}$ evaluated **at the face** ($\ell$ at face height; harmonic mean if
built from cell values). Summing the budgets telescopes the interior faces away:

$$\tau_{top} - \tau_0 = \sum_i C_d a_i U_i^2 \Delta z \qquad \text{(exact, any grid)}$$

**Momentum balance error**, used as the audit metric for every case below:

- per face: $\varepsilon_i = \tau_{i+1/2} - \tau_{i-1/2} - \Delta z\,(C_d a_i U_i^2 - dP/dx)$
- global: $\varepsilon_{tot} = \tau_{top} - \tau_0 - \sum_i C_d a_i U_i^2 \Delta z$,
  reported as a fraction of $u_*^2$.

For the nodal scheme, face fluxes for the audit are reconstructed as
$\tau_{i+1/2} = \ell^2(z_{face})\,|\Delta U/\Delta z|\,(\Delta U/\Delta z)$
from the converged solution.

## What this notebook runs

Four solution methods are compared, all of them using the **corrected mixing length**
$\ell = \kappa(z+z_0)$ near the ground, so that only the discretization and the lower
boundary condition differ:

| key | solver | grid | lower BC |
|---|---|---|---|
| `old_orig` | original nodal scheme (expanded form) | nodes $0 \dots 25$ m, $\Delta z = 0.25$ m | Dirichlet $U(0) = U_{bot}$ |
| `old_fine` | original nodal scheme | nodes $0 \dots 25$ m, $\Delta z = 0.001$ m | Dirichlet $U(0) = U_{bot}$ |
| `fdm_gm` | nodal scheme + surface conductance | cell centres $0.125 \dots 24.875$ m | $\tau_0 = g_m U_0^2$ |
| `fvm_gm` | conservative finite volume + surface conductance | cell centres, faces on the original grid | $\tau_0 = g_m U_0^2$ |

`old_fine` is the brute-force reference; for $\mathrm{LAD} = 0$ the analytic log law is
the exact reference. Every method gets the same forcing, the same drag parameters and
the same scalar sources, and each is scored with the same momentum-balance audit.

In [ ]:
%matplotlib widget
import time

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded

sns.set_context('notebook')

# --- constants, values from pyAPES.utils.constants
EPS = np.finfo(float).eps
VON_KARMAN = 0.41              # [-], von Karman constant
MOLAR_MASS_AIR = 29.0e-3       # [kg mol-1]
SPECIFIC_HEAT_AIR = 29.3       # [J mol-1 K-1]
GAS_CONSTANT_DRY_AIR = 287.05  # [J kg-1 K-1]

In [ ]:
def central_diff(y: np.ndarray, dx: float) -> np.ndarray:
    """
    Gradient dy/dx by central differences, three-point one-sided at the boundaries.
    Copied from pyAPES.utils.utilities so that this notebook is self-contained.

    Args:
        y (array): variable
        dx (float): grid increment, constant
    Returns:
        (array): dy/dx
    """
    N = len(y)
    dydx = np.ones(N) * np.nan
    dydx[1:-1] = (y[2:] - y[0:-2]) / (2 * dx)
    dydx[0] = (-3 * y[0] + 4 * y[1] - y[2]) / (2 * dx)
    dydx[-1] = (3 * y[-1] - 4 * y[-2] + y[-3]) / (2 * dx)
    return dydx

In [ ]:
def tridiag(a: np.ndarray, b: np.ndarray, C: np.ndarray, D: np.ndarray) -> np.ndarray:
    """
    Thomas algorithm for a tridiagonal system.
    Copied from pyAPES.utils.utilities.

    Args:
        a (array): subdiagonal
        b (array): diagonal
        C (array): superdiagonal
        D (array): right hand side
    Returns:
        (array): solution
    """
    n = len(a)
    V = np.zeros(n)
    G = np.zeros(n)
    U = np.zeros(n)
    x = np.zeros(n)

    V[0] = b[0]
    G[0] = C[0] / V[0]
    U[0] = D[0] / V[0]

    for i in range(1, n):
        V[i] = b[i] - a[i] * G[i - 1]
        U[i] = (D[i] - a[i] * U[i - 1]) / V[i]
        G[i] = C[i] / V[i]

    x[-1] = U[-1]
    for i in range(n - 2, -1, -1):
        x[i] = U[i] - G[i] * x[i + 1]
    return x

# Scenario definition

Everything the user is expected to touch lives in the next cell. `LAD_KIND` switches
between three canopy structures:

- `'zero'` — open surface (snow, bare field). The log law is then the exact solution
  and every diagnostic below has an analytic reference.
- `'constant'` — uniform plant area density between the ground and `h_const`
  (short vegetation, moss, grass). Note that a canopy thinner than a few grid cells is
  *not resolved* by the coarse grid: `mixing_length` falls back to the open-ground
  branch when `hc < 3*dz`. With `h_const` of the order of a few grid cells the
  original nodal solver may fail to converge altogether — that is a result, not a
  notebook bug, and the runs are flagged as `converged = False` in the tables below.
- `'weibull'` — a Weibull crown profile between `hb` and `hc` (forest). The shape is
  normalised so that $\int \mathrm{LAD}\,dz = 1$ and then scaled by `LAI`, i.e. the
  scaling factor is the leaf-area index in m$^2$ m$^{-2}$.

In [ ]:
# --- grid and surface
z_top = 25.0          # [m], top of the domain / measurement height
dz = 0.25             # [m], model grid increment
dz_fine = 0.001       # [m], brute-force reference grid increment
z0_ground = 0.001     # [m], roughness length of the surface (snow ~ 1 mm)

# --- forcing
U_ref = 5.0           # [m s-1], wind speed at z_top
u_bot = 0.01          # [m s-1], Dirichlet value at the ground for the original solver
dPdx = 0.0            # [-], u*-normalized horizontal pressure gradient

# --- canopy
LAD_KIND = 'zero'     # 'zero' | 'constant' | 'weibull'
Cd = 0.2              # [-], drag coefficient
LAI = 4.0             # [m2 m-2], leaf area index for the 'weibull' canopy
hc = 15.0             # [m], canopy height for the 'weibull' canopy
hb = 3.0              # [m], crown base height for the 'weibull' canopy
LAI_const = 0.5       # [m2 m-2], leaf area index for the 'constant' canopy
h_const = 1.0         # [m], top of the uniform layer for the 'constant' canopy

# --- scalars
Sc = 2.0              # [-], pyAPES 'Schmidt number': Ks = Sc * Km  (i.e. 1/Sc_t)
T_top = 15.0          # [degC], air temperature at z_top
RH_top = 0.7          # [-], relative humidity at z_top
CO2_top = 400.0       # [ppm], CO2 mixing ratio at z_top
P_air = 101300.0      # [Pa], ambient pressure

# canopy fluxes, distributed vertically in proportion to LAD
F_co2_canopy = -15.0  # [umol m-2 s-1], net assimilation (negative = uptake)
F_h2o_canopy = 3.0    # [mmol m-2 s-1], transpiration
F_h_canopy = 150.0    # [W m-2], sensible heat

# ground fluxes, the lower boundary condition of the scalar solvers
F_co2_ground = 3.0    # [umol m-2 s-1], soil respiration
F_h2o_ground = 0.5    # [mmol m-2 s-1], ground evaporation
F_h_ground = 20.0     # [W m-2], ground sensible heat

In [ ]:
# --- grids
# nodal grids: solution points include the ground itself
z_orig = np.arange(0.0, z_top + 0.5 * dz, dz)
z_fine = np.arange(0.0, z_top + 0.5 * dz_fine, dz_fine)

# finite-volume grid: faces on the original grid, unknowns at the cell centres
z_faces = z_orig.copy()
z_cc = 0.5 * (z_faces[:-1] + z_faces[1:])

print(f'nodal grid   : {len(z_orig)} nodes, z = {z_orig[0]:.3f} ... {z_orig[-1]:.3f} m')
print(f'fine grid    : {len(z_fine)} nodes, dz = {dz_fine} m')
print(f'FV grid      : {len(z_cc)} cells, centres {z_cc[0]:.3f} ... {z_cc[-1]:.3f} m, '
      f'faces {z_faces[0]:.2f} ... {z_faces[-1]:.2f} m')

In [ ]:
def make_lad(z: np.ndarray, kind: str, LAI: float, hc: float, hb: float = 0.0,
             h_const: float = 1.0, b: float = 0.906, c: float = 2.145) -> tuple:
    """
    Plant area density profile, normalised so that sum(LAD*dz) = LAI.

    The Weibull shape follows lad_weibul in pyAPES.utils.utilities (Teske and
    Thistle 2004, parameters of Scots pine by default).

    Args:
        z (array): [m], grid, constant increment
        kind (str): 'zero' | 'constant' | 'weibull'
        LAI (float): [m2 m-2], leaf area index, the scaling factor
        hc (float): [m], canopy height, used by 'weibull'
        hb (float): [m], crown base height, used by 'weibull'
        h_const (float): [m], top of the uniform layer, used by 'constant'
        b, c (float): Weibull shape parameters
    Returns:
        (tuple):
            lad (array): [m2 m-3], plant area density
            h_eff (float): [m], canopy height seen by the solvers
    """
    z = np.asarray(z, dtype=float)
    dz = z[1] - z[0]
    a = np.zeros(len(z))

    if kind == 'zero':
        return a, 0.0

    if kind == 'constant':
        ix = np.where((z > 0.0) & (z <= h_const))[0]
        a[ix] = 1.0
        h_eff = h_const

    elif kind == 'weibull':
        ix = np.where((z > hb) & (z <= hc))[0]
        x = np.linspace(0.0, 1.0, len(ix))       # normalized within-crown height
        a[ix] = np.abs(-(c / b) * (((1.0 - x) / b) ** (c - 1.0))
                       * np.exp(-((1.0 - x) / b) ** c)
                       / (1.0 - np.exp(-(1.0 / b) ** c)))
        h_eff = hc

    else:
        raise ValueError("kind must be 'zero', 'constant' or 'weibull'")

    a = a / (np.sum(a) * dz)                     # integral of the shape function = 1
    return LAI * a, h_eff

In [ ]:
# build LAD on each grid; the scaling factor is the leaf area index
LAI_eff = LAI_const if LAD_KIND == 'constant' else LAI

lad_orig, hc_eff = make_lad(z_orig, LAD_KIND, LAI_eff, hc, hb, h_const)
lad_fine, _ = make_lad(z_fine, LAD_KIND, LAI_eff, hc, hb, h_const)
lad_cc, _ = make_lad(z_cc, LAD_KIND, LAI_eff, hc, hb, h_const)

for name, lad, grid in (('z_orig', lad_orig, z_orig), ('z_fine', lad_fine, z_fine),
                        ('z_cc', lad_cc, z_cc)):
    integral = np.sum(lad) * (grid[1] - grid[0])
    assert abs(integral - (0.0 if LAD_KIND == 'zero' else LAI_eff)) < 1e-6, name
    print(f'{name:8}: integral(LAD dz) = {integral:.4f} m2 m-2')
print(f'canopy height seen by the solvers: hc_eff = {hc_eff} m '
      f'(coarse grid resolves a canopy only if hc_eff > 3*dz = {3*dz} m)')

In [ ]:
# --- forcing in u*-normalized form
# u_star is the scale used to normalize the solvers; the friction velocity that
# actually emerges from a solution is sqrt(tau_top) * u_star.
u_star = VON_KARMAN * U_ref / np.log(z_top / z0_ground)
Utop_n = U_ref / u_star
Ubot_n = u_bot / u_star

# molar density of air and the humidity boundary value
CF = P_air / (GAS_CONSTANT_DRY_AIR * (T_top + 273.15)) / MOLAR_MASS_AIR  # [mol m-3]
esat = 611.0 * np.exp(17.502 * T_top / (T_top + 240.97))                 # [Pa]
H2O_top = RH_top * esat / P_air                                          # [mol mol-1]

print(f'u_star scale = {u_star:.4f} m s-1, Utop/u* = {Utop_n:.3f}, Ubot/u* = {Ubot_n:.4f}')
print(f'CF = {CF:.2f} mol m-3, H2O(z_top) = {1e3*H2O_top:.2f} mmol mol-1')

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 5))
ax.plot(lad_orig, z_orig, '-', color='tab:red', label='z_orig')
ax.plot(lad_fine, z_fine, '--', color='k', lw=1, label='z_fine')
ax.plot(lad_cc, z_cc, '-.', color='tab:green', label='z_cc')
ax.set_xlabel('LAD [m$^2$ m$^{-3}$]')
ax.set_ylabel('z [m]')
ax.set_title(f"LAD_KIND = '{LAD_KIND}'")
ax.legend(frameon=False)
ax.text(0.02, 0.98, 'a)', transform=ax.transAxes, va='top')
fig.tight_layout()

# Solvers

All three momentum solvers share the mixing length and the Picard iteration, and
differ only in the discretization and in the lower boundary condition. Units: the
solvers work with $U/u_*$, so the returned `Km` has units of **metres** ($K_m/u_*$)
and `tau` is $\tau/u_*^2$; the docstring of `closure_1_model_U` in pyAPES claiming
m$^2$ s$^{-1}$ for `Km` is wrong.

**The near-ground mixing length has to match the lower boundary condition.** Both
forms below reduce to $\ell = \kappa z$ away from the surface and differ only in how
the unresolved part is regularised:

| | $\ell = \kappa(z+z_0)$ (`ground='shift'`) | $\ell = \kappa\max(z,z_0)$ (`ground='max'`) |
|---|---|---|
| with a no-slip node at $z=0$ | correct: $U = (u_*/\kappa)\ln((z+z_0)/z_0)$ vanishes at the node and has the right log asymptote | wrong: a constant-$K$ sublayer forms below $z_0$ and the emergent roughness becomes $z_0/e$ |
| with the conductance BC | inconsistent: $g_m$ was derived by integrating $\ell = \kappa z$, leaving an $O(z_0/z_{c,0})$ error in $\tau$ that *grows* when the grid is refined | correct: the discrete log law is exact to round-off at any $\Delta z$ |

The solvers therefore choose the form from their own boundary condition, and the
audit uses the form of the scheme it is auditing.

In [ ]:
def mixing_length(z: np.ndarray, h: float, d: float, z0: float,
                  l_min: float = None, dz: float = None,
                  ground: str = 'shift') -> np.ndarray:
    """
    Turbulent mixing length: linear above the canopy, constant within it and
    proportional to the height above ground below it.

    Compared with pyAPES.microclimate.micromet.mixing_length the grid-dependent
    constants are gone: the near-ground branch uses the physical roughness length
    instead of dz/2, the floor is applied with np.maximum instead of an additive
    l_min, and an unresolved canopy (h < 3*dz) falls back to the open-ground branch.

    The near-ground branch has two variants, and **the choice must match the lower
    boundary condition of the solver**:

    - `ground = 'shift'`: l = kappa*(z + z0). Use it when the grid has a node at
      z = 0 with a no-slip Dirichlet condition. The continuous solution is then
      U = (u*/kappa)*ln((z+z0)/z0), which vanishes at the node and has the correct
      log asymptote with the prescribed z0.
    - `ground = 'max'`: l = kappa*max(z, z0), i.e. plain l = kappa*z with a floor.
      Use it with the surface-conductance boundary condition, whose derivation
      integrates l = kappa*z from z0 upwards. Any other form makes the closure and
      the boundary condition inconsistent; with 'shift' the discrete log law keeps
      an O(z0/z_c0) error in tau that *grows* as the grid is refined, while with
      'max' the finite-volume solution is exact to round-off at any dz.

    Using 'max' together with a no-slip node at z = 0 is the mistake in the other
    direction: a constant-K sublayer forms below z0 and the emergent roughness
    becomes z0/e, a factor of 2.7 too small.

    References:
        Juang et al. 2008, Boundary-Layer Meteorology 128, 1-32.

    Args:
        z (array): [m], heights where l_mix is evaluated
        h (float): [m], canopy height
        d (float): [m], displacement height
        z0 (float): [m], roughness length of the ground
        l_min (float): [m], lower limit of l_mix, kappa*z0 if None
        dz (float): [m], grid increment; taken from z if None (z may be non-uniform,
            e.g. the log-mean face heights of the finite-volume scheme)
        ground (str): 'shift' | 'max', see above
    Returns:
        (array): [m], turbulent mixing length
    """
    dz = z[1] - z[0] if dz is None else dz

    if l_min is None:
        l_min = VON_KARMAN * z0

    if ground == 'shift':
        l_ground = VON_KARMAN * (z + z0)
    elif ground == 'max':
        l_ground = VON_KARMAN * z
    else:
        raise ValueError("ground must be 'shift' or 'max'")

    if h < 3 * dz:   # canopy not resolved by the grid -> open-ground branch
        return np.maximum(l_ground, l_min)

    alpha = (h - d) * VON_KARMAN / (h + EPS)
    I_F = np.sign(z - h) + 1.0
    l_mix = alpha * h * (1 - I_F / 2) + (I_F / 2) * (VON_KARMAN * (z - d))

    sc = (alpha * h) / VON_KARMAN
    ix = np.where(z < sc)
    l_mix[ix] = l_ground[ix]

    return np.maximum(l_mix, l_min)

In [ ]:
def surface_conductance(z_c0: float, z0: float) -> float:
    """
    Bulk transfer coefficient of the unresolved log layer between the aerodynamic
    surface z0 and the lowest solution point z_c0: tau_0/rho = g_m * U_0^2.

    Args:
        z_c0 (float): [m], height of the lowest solution point
        z0 (float): [m], roughness length
    Returns:
        (float): [-], g_m
    """
    if z_c0 <= z0:
        raise ValueError('lowest solution point must be above z0')
    return (VON_KARMAN / np.log(z_c0 / z0)) ** 2

In [ ]:
def emergent_z0(z: np.ndarray, U: np.ndarray, d: float = 0.0, n_fit: int = None) -> float:
    """
    Roughness length implied by a solution, from a log-profile fit to the uppermost
    tenth of the grid. Equals the input z0 only if the scheme reproduces the log law.

    Args:
        z (array): [m], solution points
        U (array): [-], u*-normalized wind speed
        d (float): [m], displacement height
        n_fit (int): number of uppermost points used in the fit
    Returns:
        (float): [m], emergent roughness length
    """
    n_fit = max(3, len(z) // 10) if n_fit is None else n_fit
    zz = z[-n_fit:] - d
    ok = zz > 0.0
    if ok.sum() < 2:
        return np.nan
    slope, intercept = np.polyfit(np.log(zz[ok]), U[-n_fit:][ok], deg=1)
    return float(np.exp(-intercept / slope)) if slope > 0 else np.nan

In [ ]:
def closure_1_model_u_original(z: np.ndarray, z0: float, Cd: float, lad: np.ndarray,
                               hc: float, Utop: float, Ubot: float = 0.0,
                               dPdx: float = 0.0, lbc: str = 'dirichlet',
                               U_ini: np.ndarray = None, l_min: float = None,
                               max_iter: int = 200, tol: float = 1e-6,
                               relax: float = 0.5) -> dict:
    """
    The original pyAPES nodal solver: the expanded form -Km*U'' - Km'*U' + Cd*a*U^2
    = dPdx discretized at the nodes, Picard iteration on a tridiagonal system.

    Two bugs of closure_1_model_U are fixed here (dz = z[1] - z[0] instead of
    z[1] - z[2], and the boundary switch tested with a string instead of a bool
    cast); the mixing length is the corrected one. The discretization itself is
    untouched, so this is the reference for "what the model does today".

    Args:
        z (array): [m], nodes, constant increment, z[0] = ground
        z0 (float): [m], roughness length
        Cd (float): [-], drag coefficient
        lad (array): [m2 m-3], one-sided plant area density
        hc (float): [m], canopy height
        Utop (float): [-], U/u* at the top node
        Ubot (float): [-], U/u* at the ground node, used if lbc = 'dirichlet'
        dPdx (float): [-], u*-normalized pressure gradient
        lbc (str): 'dirichlet' | 'flux' (zero flux)
        U_ini (array): [-], initial guess
        l_min (float): [m], minimum mixing length, kappa*z0 if None
        max_iter (int): maximum number of Picard iterations
        tol (float): [-], convergence criterion max|U(n+1) - U(n)|
        relax (float): [-], relaxation weight of the new iterate
    Returns:
        (dict): z, U, tau, tau_surface, Km, l_mix, d, gm, grid, iterations, err,
            converged
    """
    z = np.asarray(z, dtype=float)
    lad = 0.5 * np.asarray(lad, dtype=float)     # frontal area density
    dz = z[1] - z[0]
    N = len(z)

    if l_min is None:
        l_min = VON_KARMAN * z0

    U = np.linspace(max(Ubot, EPS), Utop, N) if U_ini is None else U_ini.copy()
    err, iter_no = 1e9, 0

    while err > tol and iter_no < max_iter:
        iter_no += 1

        Fd = Cd * lad * U ** 2                   # drag force
        d = np.sum(z * Fd) / (np.sum(Fd) + EPS)  # displacement height
        l_mix = mixing_length(z, hc, d, z0=z0, l_min=l_min, ground='shift')

        y = central_diff(U, dz)
        Km = l_mix ** 2 * np.abs(y)

        a1 = -Km
        a2 = central_diff(-Km, dz)
        a3 = Cd * lad * U                        # Picard linearization of the drag

        lod = a1 / dz ** 2 - a2 / (2 * dz)
        dia = -2.0 * a1 / dz ** 2 + a3
        upd = a1 / dz ** 2 + a2 / (2 * dz)
        rhs = np.full(N, dPdx)

        lod[-1], dia[-1], upd[-1], rhs[-1] = 0.0, 1.0, 0.0, Utop
        if lbc == 'dirichlet':
            lod[0], dia[0], upd[0], rhs[0] = 0.0, 1.0, 0.0, Ubot
        elif lbc == 'flux':
            lod[0], dia[0], upd[0], rhs[0] = 0.0, 1.0, -1.0, 0.0
        else:
            raise ValueError("lbc must be 'dirichlet' or 'flux'")

        Un = tridiag(lod, dia, upd, rhs)
        err = np.max(np.abs(Un - U))
        U = relax * Un + (1.0 - relax) * U

    y = central_diff(U, dz)
    l_mix = mixing_length(z, hc, d, z0=z0, l_min=l_min, ground='shift')
    Km = l_mix ** 2 * np.abs(y)
    tau = Km * y                                 # positive = downward flux

    return {'z': z, 'U': U, 'tau': tau, 'tau_surface': tau[0], 'Km': Km,
            'l_mix': l_mix, 'd': d, 'gm': np.nan, 'grid': 'node', 'l_form': 'shift',
            'iterations': iter_no, 'err': err, 'converged': err <= tol}

In [ ]:
def closure_1_model_u_fdm(z: np.ndarray, z0: float, Cd: float, lad: np.ndarray,
                          hc: float, Utop: float, Ubot: float = 0.0,
                          dPdx: float = 0.0, lbc: str = 'conductance',
                          gm: float = None, U_ini: np.ndarray = None,
                          l_min: float = None, max_iter: int = 200,
                          tol: float = 1e-6, relax: float = 0.5) -> dict:
    """
    The same nodal discretization as closure_1_model_u_original, but the unknowns
    sit at cell centres and the lower boundary condition is the surface conductance:
    the budget of the lowest cell receives tau_0 = g_m*U_0^2 as a Picard-linearized
    sink, structurally identical to the canopy drag term.

    Comparing this with closure_1_model_u_fvm separates the effect of the boundary
    condition from the effect of the discretization.

    Args:
        z (array): [m], cell centres, constant increment
        z0 (float): [m], roughness length
        Cd (float): [-], drag coefficient
        lad (array): [m2 m-3], one-sided plant area density
        hc (float): [m], canopy height
        Utop (float): [-], U/u* at the uppermost cell centre
        Ubot (float): [-], U/u* at the lowest cell, used if lbc = 'dirichlet'
        dPdx (float): [-], u*-normalized pressure gradient
        lbc (str): 'conductance' | 'dirichlet' | 'flux'
        gm (float): [-], surface conductance; from z[0] and z0 if None.
            gm -> inf reproduces no-slip, gm -> 0 the zero-flux branch.
        U_ini (array): [-], initial guess
        l_min (float): [m], minimum mixing length, kappa*z0 if None
        max_iter (int): maximum number of Picard iterations
        tol (float): [-], convergence criterion
        relax (float): [-], relaxation weight
    Returns:
        (dict): as closure_1_model_u_original
    """
    z = np.asarray(z, dtype=float)
    lad = 0.5 * np.asarray(lad, dtype=float)
    dz = z[1] - z[0]
    N = len(z)

    if l_min is None:
        l_min = VON_KARMAN * z0
    # the near-ground mixing length must match the boundary condition:
    # the conductance is derived by integrating l = kappa*z from z0 upwards
    l_form = 'max' if lbc == 'conductance' else 'shift'
    if lbc == 'conductance' and gm is None:
        gm = surface_conductance(z[0], z0)

    U = np.linspace(max(Ubot, EPS), Utop, N) if U_ini is None else U_ini.copy()
    err, iter_no = 1e9, 0

    while err > tol and iter_no < max_iter:
        iter_no += 1

        Fd = Cd * lad * U ** 2
        d = np.sum(z * Fd) / (np.sum(Fd) + EPS)
        l_mix = mixing_length(z, hc, d, z0=z0, l_min=l_min, ground=l_form)

        y = central_diff(U, dz)
        Km = l_mix ** 2 * np.abs(y)

        a1 = -Km
        a2 = central_diff(-Km, dz)
        a3 = Cd * lad * U

        lod = a1 / dz ** 2 - a2 / (2 * dz)
        dia = -2.0 * a1 / dz ** 2 + a3
        upd = a1 / dz ** 2 + a2 / (2 * dz)
        rhs = np.full(N, dPdx)

        lod[-1], dia[-1], upd[-1], rhs[-1] = 0.0, 1.0, 0.0, Utop

        if lbc == 'dirichlet':
            lod[0], dia[0], upd[0], rhs[0] = 0.0, 1.0, 0.0, Ubot
        elif lbc == 'flux':
            lod[0], dia[0], upd[0], rhs[0] = 0.0, 1.0, -1.0, 0.0
        elif lbc == 'conductance':
            # budget of the lowest cell:
            #   -(tau_face - tau_0)/dz + Cd*a_0*U_0^2 = dPdx
            #   tau_face = K_face*(U_1 - U_0)/dz,  tau_0 = g_m*U_0^2
            l_face = mixing_length(z + 0.5 * dz, hc, d, z0=z0, l_min=l_min,
                                   ground=l_form)[0]
            K_face = l_face ** 2 * np.abs(U[1] - U[0]) / dz
            lod[0] = 0.0
            dia[0] = K_face / dz ** 2 + gm * U[0] / dz + Cd * lad[0] * U[0]
            upd[0] = -K_face / dz ** 2
            rhs[0] = dPdx
        else:
            raise ValueError("lbc must be 'conductance', 'dirichlet' or 'flux'")

        Un = tridiag(lod, dia, upd, rhs)
        err = np.max(np.abs(Un - U))
        U = relax * Un + (1.0 - relax) * U

    y = central_diff(U, dz)
    l_mix = mixing_length(z, hc, d, z0=z0, l_min=l_min, ground=l_form)
    Km = l_mix ** 2 * np.abs(y)
    tau = Km * y

    if lbc == 'conductance':
        tau_surface = gm * U[0] ** 2
    elif lbc == 'flux':
        tau_surface = 0.0
    else:
        tau_surface = tau[0]

    return {'z': z, 'U': U, 'tau': tau, 'tau_surface': tau_surface, 'Km': Km,
            'l_mix': l_mix, 'd': d, 'gm': gm, 'grid': 'cell', 'l_form': l_form,
            'iterations': iter_no, 'err': err, 'converged': err <= tol}

In [ ]:
def closure_1_model_u_fvm(z_f: np.ndarray, z0: float, Cd: float, lad: np.ndarray,
                          hc: float, Utop: float, Ubot: float = 0.0,
                          dPdx: float = 0.0, lbc: str = 'conductance',
                          gm: float = None, U_ini: np.ndarray = None,
                          l_min: float = None, max_iter: int = 200,
                          tol: float = 1e-6, relax: float = 0.5) -> dict:
    """
    Conservative finite-volume solver. Faces are the original model grid, unknowns
    are at the cell centres, and the face fluxes

        tau_(i+1/2) = K_(i+1/2) * (U_(i+1) - U_i)/dz

    are shared by the two adjacent cell budgets, so the interior faces telescope and
    tau_top - tau_0 = sum(Cd*a*U^2)*dz holds exactly on any grid.

    The face mixing length is evaluated at the **log-mean** height of the two
    neighbouring cell centres, z_lm = dz / ln(z_(i+1)/z_i), instead of their
    arithmetic mean. In the log layer the discrete gradient (U_(i+1)-U_i)/dz is the
    exact gradient at z_lm, so this choice makes the discrete log law an exact
    solution; the arithmetic mean leaves a ~2 % bias in tau at any resolution. Above
    the first few cells the two coincide to O((dz/z)^2).

    Args:
        z_f (array): [m], cell faces, constant increment, z_f[0] = ground
        z0 (float): [m], roughness length
        Cd (float): [-], drag coefficient
        lad (array): [m2 m-3], one-sided plant area density at the cell centres
        hc (float): [m], canopy height
        Utop (float): [-], U/u* at the uppermost cell centre
        Ubot (float): [-], U/u* at the lowest cell, used if lbc = 'dirichlet'
        dPdx (float): [-], u*-normalized pressure gradient
        lbc (str): 'conductance' | 'dirichlet' | 'flux'
        gm (float): [-], surface conductance; from the lowest cell centre if None
        U_ini (array): [-], initial guess
        l_min (float): [m], minimum mixing length
        max_iter (int): maximum number of Picard iterations
        tol (float): [-], convergence criterion
        relax (float): [-], relaxation weight
    Returns:
        (dict): as closure_1_model_u_original, plus
            z_f (array): [m], cell faces
            tau_face (array): [-], momentum flux through every face, tau_face[0]
                is the flux into the ground
    """
    z_f = np.asarray(z_f, dtype=float)
    z = 0.5 * (z_f[:-1] + z_f[1:])               # cell centres
    lad = 0.5 * np.asarray(lad, dtype=float)
    dz = z_f[1] - z_f[0]
    N = len(z)

    if l_min is None:
        l_min = VON_KARMAN * z0
    # see mixing_length: 'max' is the form consistent with the conductance
    l_form = 'max' if lbc == 'conductance' else 'shift'
    if lbc == 'conductance' and gm is None:
        gm = surface_conductance(z[0], z0)

    # log-mean face heights for the interior faces
    z_eff = z_f.copy()
    z_eff[1:-1] = dz / np.log(z[1:] / z[:-1])

    U = np.linspace(max(Ubot, EPS), Utop, N) if U_ini is None else U_ini.copy()
    err, iter_no = 1e9, 0

    while err > tol and iter_no < max_iter:
        iter_no += 1

        Fd = Cd * lad * U ** 2
        d = np.sum(z * Fd) / (np.sum(Fd) + EPS)

        l_face = mixing_length(z_eff, hc, d, z0=z0, l_min=l_min, dz=dz,
                               ground=l_form)
        dUdz_f = np.zeros(N + 1)
        dUdz_f[1:-1] = (U[1:] - U[:-1]) / dz
        K_f = l_face ** 2 * np.abs(dUdz_f)

        lod = -K_f[:-1] / dz ** 2
        dia = (K_f[:-1] + K_f[1:]) / dz ** 2 + Cd * lad * U
        upd = -K_f[1:] / dz ** 2
        rhs = np.full(N, dPdx)

        lod[-1], dia[-1], upd[-1], rhs[-1] = 0.0, 1.0, 0.0, Utop

        if lbc == 'dirichlet':
            lod[0], dia[0], upd[0], rhs[0] = 0.0, 1.0, 0.0, Ubot
        elif lbc == 'flux':
            lod[0], dia[0], upd[0], rhs[0] = 0.0, 1.0, -1.0, 0.0
        elif lbc == 'conductance':
            lod[0] = 0.0
            dia[0] = K_f[1] / dz ** 2 + gm * U[0] / dz + Cd * lad[0] * U[0]
            upd[0] = -K_f[1] / dz ** 2
            rhs[0] = dPdx
        else:
            raise ValueError("lbc must be 'conductance', 'dirichlet' or 'flux'")

        Un = tridiag(lod, dia, upd, rhs)
        err = np.max(np.abs(Un - U))
        U = relax * Un + (1.0 - relax) * U

    l_face = mixing_length(z_eff, hc, d, z0=z0, l_min=l_min, dz=dz,
                           ground=l_form)
    dUdz_f = np.zeros(N + 1)
    dUdz_f[1:-1] = (U[1:] - U[:-1]) / dz
    dUdz_f[-1] = dUdz_f[-2]
    K_f = l_face ** 2 * np.abs(dUdz_f)
    tau_face = K_f * dUdz_f

    if lbc == 'conductance':
        tau_face[0] = gm * U[0] ** 2
    elif lbc == 'flux':
        tau_face[0] = 0.0
    else:
        tau_face[0] = tau_face[1]

    return {'z': z, 'z_f': z_f, 'U': U,
            'tau': 0.5 * (tau_face[:-1] + tau_face[1:]),
            'tau_face': tau_face, 'tau_surface': tau_face[0],
            'Km': 0.5 * (K_f[:-1] + K_f[1:]),
            'l_mix': mixing_length(z, hc, d, z0=z0, l_min=l_min, ground=l_form),
            'd': d, 'gm': gm, 'grid': 'cell', 'l_form': l_form,
            'iterations': iter_no, 'err': err, 'converged': err <= tol}

In [ ]:
def mixing_length_opa(z,z0, hc, d, dz=None, l_min = None):

        dz = z[1] - z[0] if dz is None else dz

        if l_min is None:
            l_min = VON_KARMAN * z0

        l_ground = VON_KARMAN * z

        if hc < 3 * dz:   # canopy not resolved by the grid -> open-ground branch
            return np.maximum(l_ground, l_min)

        alpha = (hc - d) * VON_KARMAN / (hc + EPS)
        I_F = np.sign(z - hc) + 1.0
        l_mix = alpha * hc * (1 - I_F / 2) + (I_F / 2) * (VON_KARMAN * (z - d))

        sc = (alpha * hc) / VON_KARMAN
        ix = np.where(z < sc)
        l_mix[ix] = l_ground[ix]

        return np.maximum(l_mix, l_min)

In [ ]:
def closure_1_model_U_fvm_opa(z: np.ndarray, z0: float, Cd: float, lad: np.ndarray,
                              hc: float, Utop: float, Ubot: float = 0.0,
                              dPdx: float = 0.0, lbc: str = 'conductance', gamma: float = 0.5,
                              l_min: float = None, max_iter: int = 200):
    z_faces = z.copy()  # z input is faces
    z_midpoint = 0.5 * (z_faces[:-1] + z_faces[1:]) # z_midpoint has shape (N-1,).
     
    lad_faces = 0.5 * np.asarray(lad, dtype=float)
    lad = 0.5 * (lad_faces[:-1] + lad_faces[1:])
    dz = z_midpoint[1] - z_midpoint[0]

    N = len(z_midpoint)

    if l_min is None:
        l_min = VON_KARMAN * z0

    if lbc == 'conductance':
        g_tau = (VON_KARMAN / np.log(z_midpoint[0] / z0)) ** 2
    else:
        raise NotImplementedError("only lbc='conductance' is implemented")

    # Initial guess for U profile
    U = np.linspace(max(Ubot, EPS), Utop, N) 
    err = 1e6
    iter_no = 0

    # Define RHS of the matrix equation outside iteration loop as this never changes
    rhs = np.full(N, -1.0*dPdx) # This needs to be minus if we have vertical pressure gradient. In OPT notes we only deal with system where dP/dx=0
    rhs[-1] = Utop
    while err > 1e-6 and iter_no < max_iter:
        iter_no += 1
        
        Fd = Cd * lad * U ** 2 # drag force from last iteration
        d = np.sum(z_midpoint * Fd) / (np.sum(Fd) + EPS) # displacement height as centroid of drag force
        l_mix = mixing_length_opa(z_faces[1:-1], z0, hc, d, l_min=l_min) # mixing length at cell faces (not midpoints since flux is at faces)
  
        dUdz = (U[1:] - U[:-1]) / dz # calculate dUdz at elements which have element upwards and downwards from them
        #dUdz[0] = (U[0]) / dz0 # calculate dUdz at the bottom flux. Here we assume U(z0) = 0
        #dUdz[-1] = (Utop - U[-1]) / dz # calculate dUdz at the top flux
        Km = l_mix ** 2 * np.abs(dUdz)
        
        #Km_mean = 2 * Km[:-1] * Km[1:] / (Km[:-1] + Km[1:] + EPS) # harmonic mean of Km_i and Km_(i +/- 1), shape (N-1,)
        
        A_plus = Km[1:] / dz ** 2 # Check OPT notes from 25.8.2026 for naming of these matrix elements, shape (N-2,)
        A_minus = Km[:-1] / dz ** 2 # shape (N-2,)
        B = -A_plus - A_minus - Cd*lad[1:-1]*U[1:-1] # shape (N-2)
        C = -Km[0]/dz**2 - g_tau/dz*U[0]-Cd*lad[0]*U[0] # shape (1,)

        # Build tridiagonal matrix for solve_banded
        ab = np.zeros((3, N))
        ab[0, 2:] = A_plus  # upper diagonal
        ab[0, 1] = Km[0]/dz**2 # upper element for the conductance BC 
        ab[1, 1:-1] = B # main diagonal
        ab[1, 0] = C # conductance BC at bottom
        ab[1,-1] = 1 # Dirichlet BC at top
        ab[2, :-2] = A_minus # lower diagonal

        
        U_new = solve_banded((1, 1), ab, rhs)
        err = np.max(np.abs(U_new - U))
        U = gamma * U_new + (1.0 - gamma) * U


    l_mix= mixing_length_opa(z_faces[1:-1], z0, hc, d, l_min=l_min)
    dUdz = (U[1:] - U[:-1]) / dz
    Km_int = l_mix ** 2 * np.abs(dUdz) #Km at interior faces

    # Create face arrays at original z which is what we want to return
    tau_f = np.zeros(N+1)
    Km_f = np.zeros_like(tau_f)
    U_f = np.zeros_like(tau_f)

    tau_f[1:-1]= Km_int * dUdz
    Km_f[1:-1] = Km_int
    U_f[1:-1] = 0.5*(U[1:] + U[:-1]) # mean between two adjacent cell centers

    #ground face: the conductance carries the whole unresolved layer from z0 to z[0]
    #Km_f[0] is the diffusivity that reprocudes tau_0 in this layer
    #since Km_f[0] (U[0]-0)/dz0 = g_tau U[0]**2 <=> Km_f[0] = g_tau U[0] dz0
    dz0 = z_midpoint[0] - z0
    tau_f[0] = g_tau*U[0]**2
    Km_f[0] = g_tau*U[0]*dz0
    U_f[0] = 0.0 #U(z0) = 0, here we approzimate that U(z=0) = U(z0) = 0

    # top face: Dirichlet BC
    tau_f[-1] = tau_f[-2]
    Km_f[-1] = Km_f[-2]
    U_f[-1] = U[-1] # Dirichlet BC at top


    # Return with z_midpoint as z (cell centres) and z_faces as z_f (faces)
    # to match the convention of closure_1_model_u_fvm.
    # K_f/tau_face already have length N (one value per cell centre) in this
    # formulation, so they are used directly instead of averaging them down
    # to N-1, which would break the length match with z_midpoint (length N).
    return {'z': z_midpoint, 'z_f': z_faces, 'U': U, 'U_f': U_f,
            'tau': tau_f,
            'tau_f': tau_f, 'tau_surface': tau_f[0],
            'Km_f': Km_f, 'l_mix_f': l_mix,
            'l_mix': mixing_length_opa(z_midpoint, z0, hc, d, l_min=l_min),
            'd': d, 'gm': g_tau, 'grid': 'cell', 'l_form': 'max',
            'iterations': iter_no, 'err': err, 'converged': err <= 1e-6}

## Momentum balance audit

Every converged profile is scored with the same flux-form yardstick, whatever
discretization produced it: face fluxes are reconstructed as
$\tau_{i+1/2} = \ell^2(z_{lm})\,|\Delta U/\Delta z|\,(\Delta U/\Delta z)$
(log-mean face height, as in the FVM solver), and each cell budget leaves a residual

$$\varepsilon_i = \tau_{i+1/2} - \tau_{i-1/2} - \Delta z\,(C_d a_i U_i^2 - dP/dx),
\qquad
\varepsilon_{tot} = \tau_{top} - \tau_{bottom} - \sum_i C_d a_i U_i^2 \Delta z$$

reported as a fraction of the momentum flux at the uppermost audited face. For the
conductance runs $\tau_{bottom}$ is the ground flux $g_m U_0^2$; for the nodal runs
there is no flux at the ground node, so $\tau_{bottom}$ is the flux through the
lowest interior face and the audit covers the cells above it.

In [ ]:
def momentum_balance(res: dict, z0: float, Cd: float, lad: np.ndarray,
                     hc: float, dPdx: float = 0.0) -> dict:
    """
    Flux-form momentum balance of a converged profile.

    Two branches. A solver that publishes its own face fluxes under 'tau_f'
    (on the face grid 'z_f') is audited with them, so the residual measures the
    cell budgets the solver actually solved. Everything else has its face fluxes
    reconstructed from the profile with the notebook's mixing length.

    Args:
        res (dict): output of any of the momentum solvers
        z0 (float): [m], roughness length
        Cd (float): [-], drag coefficient
        lad (array): [m2 m-3], one-sided plant area density on res['z']
        hc (float): [m], canopy height
        dPdx (float): [-], u*-normalized pressure gradient
    Returns:
        (dict): as before
    """
    z, U = res['z'], res['U']
    dz = z[1] - z[0]
    lad_f = 0.5 * np.asarray(lad, dtype=float)

    if 'tau_f' in res:
        # --- native faces: cell i is bounded by faces i and i+1
        tau_face = np.asarray(res['tau_f'], dtype=float)
        z_face = np.asarray(res['z_f'], dtype=float)
        idx = np.arange(0, len(z) - 1)          # topmost cell is Dirichlet
        sink = dz * (Cd * lad_f[idx] * U[idx] ** 2 - dPdx)
        eps = tau_face[idx + 1] - tau_face[idx] - sink
        tau_bottom, tau_top = tau_face[0], tau_face[idx[-1] + 1]
        eps_tot = tau_top - tau_bottom - np.sum(sink)
        tau_ref = abs(tau_top) + EPS
        return {'z': z[idx], 'eps': eps, 'eps_rel': eps / tau_ref,
                'eps_tot': eps_tot, 'eps_tot_rel': eps_tot / tau_ref,
                'z_face': z_face, 'tau_face': tau_face,
                'tau_bottom': tau_bottom, 'tau_top': tau_top}

    # --- reconstructed faces: log-mean of the neighbouring solution points
    z_face = 0.5 * (z[:-1] + z[1:])
    ok = z[:-1] > 0.0
    z_face[ok] = dz / np.log(z[1:][ok] / z[:-1][ok])

    l_face = mixing_length(z_face, hc, res['d'], z0=z0,
                           l_min=VON_KARMAN * z0, dz=dz,
                           ground=res.get('l_form', 'shift'))
    dUdz = (U[1:] - U[:-1]) / dz
    tau_face = l_face ** 2 * np.abs(dUdz) * dUdz

    if res['grid'] == 'cell':
        tau_lo = np.concatenate(([res['tau_surface']], tau_face[:-1]))
        tau_hi = tau_face
        idx = np.arange(0, len(z) - 1)
    else:
        tau_lo = tau_face[:-1]
        tau_hi = tau_face[1:]
        idx = np.arange(1, len(z) - 1)

    sink = dz * (Cd * lad_f[idx] * U[idx] ** 2 - dPdx)
    eps = tau_hi - tau_lo - sink
    eps_tot = tau_hi[-1] - tau_lo[0] - np.sum(sink)
    tau_ref = abs(tau_hi[-1]) + EPS

    return {'z': z[idx], 'eps': eps, 'eps_rel': eps / tau_ref,
            'eps_tot': eps_tot, 'eps_tot_rel': eps_tot / tau_ref,
            'z_face': z_face, 'tau_face': tau_face,
            'tau_bottom': tau_lo[0], 'tau_top': tau_hi[-1]}


# Scalar transport

The scalar equation is the same diffusion problem with $K_s = Sc\,K_m$ (pyAPES
convention, `Sc = 2.0`), a Dirichlet value at the top and a prescribed flux at the
ground. Two formulations are compared, one per family of momentum solvers:

- `closure_1_model_scalar_nodal` — what pyAPES does today. The ground flux is turned
  into a gradient between the two lowest nodes, $C_0 - C_1 = F\Delta z/(\rho_{mol}K_{1/2})$,
  so the *resolved* diffusivity carries the whole surface resistance and the budget of
  node 0 (with its source term) is discarded.
- `closure_1_model_scalar_fvm` — the ground flux enters the budget of the lowest cell
  directly, so no diffusivity has to be invented at the surface. The surface value
  itself follows from the analytic jump across the unresolved layer,
  $C_{surface} = C_0 + F/(\rho_{mol}\,g_s U_0)$ with $g_s = g_m Sc$ — this is the
  number the snow and forest-floor schemes need.

In [ ]:
def closure_1_model_scalar_nodal(dz: float, Ks: np.ndarray, source: np.ndarray,
                                 ubc: float, lbc: float, CF: float) -> np.ndarray:
    """
    Steady-state scalar profile, pyAPES formulation (closure_1_model_scalar).

    Note: the lower boundary is a gradient condition through the resolved face
    diffusivity Ks_(1/2), and node 0 has no budget of its own, so source[0] does not
    affect the solution.

    Args:
        dz (float): [m], grid increment
        Ks (array): [m2 s-1], eddy diffusivity at the nodes
        source (array): [mol m-3 s-1], source, positive = release into the air
        ubc (float): [mol mol-1], mixing ratio at the top node
        lbc (float): [mol m-2 s-1], ground flux, positive upward
        CF (float): [mol m-3], molar density of air
    Returns:
        (array): [mol mol-1], scalar profile
    """
    N = len(Ks)
    Ksf = np.empty(N + 1)                        # Ksf[i] = face below node i
    Ksf[1:-1] = 0.5 * (Ks[:-1] + Ks[1:])
    Ksf[0], Ksf[-1] = Ks[0], Ks[-1]

    a, b, g, f = (np.zeros(N) for _ in range(4))

    a[1:-1] = Ksf[1:-2]
    b[1:-1] = -(Ksf[1:-2] + Ksf[2:-1])
    g[1:-1] = Ksf[2:-1]
    f[1:-1] = -source[1:-1] / CF * dz ** 2

    a[-1], b[-1], g[-1], f[-1] = 0.0, 1.0, 0.0, ubc            # Dirichlet at the top
    a[0], b[0], g[0], f[0] = 0.0, 1.0, -1.0, (lbc / CF) * dz / (Ksf[1] + EPS)

    return tridiag(a, b, g, f)

In [ ]:
def closure_1_model_scalar_fvm(dz: float, Ks: np.ndarray, source: np.ndarray,
                               ubc: float, lbc: float, CF: float) -> np.ndarray:
    """
    Steady-state scalar profile, conservative cell-centred formulation. Face
    diffusivities are harmonic means of the cell values (resistances in series), and
    the ground flux enters the budget of the lowest cell directly.

    Args and returns as in closure_1_model_scalar_nodal, Ks at the cell centres.
    """
    N = len(Ks)
    Kf = np.zeros(N + 1)
    Kf[1:-1] = 2.0 * Ks[:-1] * Ks[1:] / (Ks[:-1] + Ks[1:] + EPS)   # harmonic mean

    a, b, g, f = (np.zeros(N) for _ in range(4))

    a[1:-1] = -Kf[1:-2] / dz
    b[1:-1] = (Kf[1:-2] + Kf[2:-1]) / dz
    g[1:-1] = -Kf[2:-1] / dz
    f[1:-1] = source[1:-1] * dz / CF

    a[-1], b[-1], g[-1], f[-1] = 0.0, 1.0, 0.0, ubc            # Dirichlet at the top

    a[0] = 0.0                                                 # ground flux BC
    b[0] = Kf[1] / dz
    g[0] = -Kf[1] / dz
    f[0] = (source[0] * dz + lbc) / CF

    return tridiag(a, b, g, f)

In [ ]:
def scalar_surface_value(C0, F, gs, U0, CF):
    """
    Scalar value at the surface from the analytic jump across the unresolved
    layer between z0 and the lowest cell centre.

        C_surface = C0 + F / (rho_mol * gs * U0),    gs = gm * Sc

    Args:
        C0 (float): value at the lowest cell centre, same units as the profile
        F (float): [mol m-2 s-1], ground flux, positive upward
        gs (float): [-], surface conductance for the scalar
        U0 (float): [m s-1], wind speed at the lowest cell centre
        CF (float): [mol m-3], molar concentration of air
    Returns:
        (float): value at z0
    """
    return C0 + F / (CF * gs * U0 + EPS)


In [ ]:
def canopy_sources(z: np.ndarray, lad: np.ndarray) -> dict:
    """
    Distribute the prescribed canopy fluxes vertically in proportion to LAD.

    Args:
        z (array): [m], solution points
        lad (array): [m2 m-3], plant area density
    Returns:
        (dict): 'CO2' [mol m-3 s-1], 'H2O' [mol m-3 s-1], 'T' [mol m-3 s-1]
            (the heat source is already divided by the heat capacity of air)
    """
    dz_local = z[1] - z[0]
    total = np.sum(lad) * dz_local
    shape = lad / total if total > 0 else np.zeros(len(z))     # [m-1]
    return {'CO2': 1e-6 * F_co2_canopy * shape,
            'H2O': 1e-3 * F_h2o_canopy * shape,
            'T': F_h_canopy / SPECIFIC_HEAT_AIR * shape}

In [ ]:
def run_case(name: str, kind: str, z_grid: np.ndarray, lad: np.ndarray) -> dict:
    t0 = time.perf_counter()
    if kind == 'old':
        res = closure_1_model_u_original(z_grid, z0_ground, Cd, lad, hc_eff,
                                         Utop=Utop_n, Ubot=Ubot_n, dPdx=dPdx)
    elif kind == 'fdm':
        res = closure_1_model_u_fdm(z_grid, z0_ground, Cd, lad, hc_eff,
                                    Utop=Utop_n, dPdx=dPdx)
    elif kind == 'fvm':
        res = closure_1_model_u_fvm(z_grid, z0_ground, Cd, lad, hc_eff,
                                    Utop=Utop_n, dPdx=dPdx)
    elif kind == 'fvm_opa':
        res = closure_1_model_U_fvm_opa(z_grid, z0_ground, Cd, lad, hc_eff,
                                        Utop=Utop_n, dPdx=dPdx)
        # the solver takes z_grid as faces and solves at the cell centres; use the
        # same cell-centre plant area density it used internally
        lad = 0.5 * (np.asarray(lad, dtype=float)[:-1] + np.asarray(lad, dtype=float)[1:])
        # cell-centre Km and tau from the face values, so that every profile in
        # this notebook lives on res['z'] like the other 'cell' solvers
        res['Km'] = 0.5 * (res['Km_f'][:-1] + res['Km_f'][1:])
        res['tau'] = 0.5 * (res['tau_f'][:-1] + res['tau_f'][1:])
    else:
        raise ValueError("kind must be 'old', 'fdm', 'fvm' or 'fvm_opa'")
    res['wall_time'] = time.perf_counter() - t0
    res['name'] = name
    res['lad'] = lad

    z = res['z']
    dz_local = z[1] - z[0]
    res['U_dim'] = res['U'] * u_star
    res['Km_dim'] = res['Km'] * u_star
    res['tau_dim'] = res['tau'] * u_star ** 2
    res['ustar_local'] = np.sqrt(np.abs(res['tau'])) * u_star

    # profiles on the pyAPES grid (the solver faces), the deliverable of fvm_opa
    if 'U_f' in res:
        res['z_pyapes'] = res['z_f']
        res['U_pyapes'] = res['U_f'] * u_star
        res['Km_pyapes'] = res['Km_f'] * u_star
        res['tau_pyapes'] = res['tau_f'] * u_star ** 2

    res['mbe'] = momentum_balance(res, z0_ground, Cd, lad, hc_eff, dPdx)
    res['z0_emergent'] = emergent_z0(z, res['U'], res['d'])

    Ks = np.maximum(Sc * res['Km_dim'], 1e-6)
    src = canopy_sources(z, lad)
    solver = (closure_1_model_scalar_nodal if res['grid'] == 'node'
              else closure_1_model_scalar_fvm)

    co2 = solver(dz_local, Ks, src['CO2'], 1e-6 * CO2_top, 1e-6 * F_co2_ground, CF)
    h2o = solver(dz_local, Ks, src['H2O'], H2O_top, 1e-3 * F_h2o_ground, CF)
    tair = solver(dz_local, Ks, src['T'], T_top, F_h_ground / SPECIFIC_HEAT_AIR, CF)

    res['CO2'] = 1e6 * co2
    res['H2O'] = 1e3 * h2o
    res['T'] = tair

    if res['grid'] == 'cell':
        gs = res['gm'] * Sc
        res['gs'] = gs
        res['CO2_surf'] = 1e6 * scalar_surface_value(co2[0], 1e-6 * F_co2_ground,
                                                     gs, res['U_dim'][0], CF)
        res['H2O_surf'] = 1e3 * scalar_surface_value(h2o[0], 1e-3 * F_h2o_ground,
                                                     gs, res['U_dim'][0], CF)
        res['T_surf'] = scalar_surface_value(tair[0], F_h_ground / SPECIFIC_HEAT_AIR,
                                             gs, res['U_dim'][0], CF)
    else:
        res['gs'] = np.nan
        res['CO2_surf'] = res['CO2'][0]
        res['H2O_surf'] = res['H2O'][0]
        res['T_surf'] = res['T'][0]

    return res


# Acceptance tests

These run before any figure is produced. Tolerances differ by design: the finite
volume scheme is required to be exact, the nodal scheme with the conductance
boundary condition only to be close, because its residual is a property of the
discretization and not of the boundary condition.

In [ ]:
# open ground, where the log law is the exact solution
z_test_f = np.arange(0.0, z_top + 0.5 * dz, dz)
z_test_c = 0.5 * (z_test_f[:-1] + z_test_f[1:])
lad_test = np.zeros(len(z_test_c))
Utop_test = np.log(z_test_c[-1] / z0_ground) / VON_KARMAN
U0_exact = np.log(z_test_c[0] / z0_ground) / VON_KARMAN

t_fvm = closure_1_model_u_fvm(z_test_f, z0_ground, Cd, lad_test, 0.0, Utop=Utop_test)
t_fdm = closure_1_model_u_fdm(z_test_c, z0_ground, Cd, lad_test, 0.0, Utop=Utop_test)

# 1) lowest cell reproduces the analytic log-layer integral
err_fvm = abs(t_fvm['U'][0] / U0_exact - 1.0)
err_fdm = abs(t_fdm['U'][0] / U0_exact - 1.0)
assert err_fvm < 1e-6, err_fvm
assert err_fdm < 2e-2, err_fdm

# 2) FVM face fluxes are flat and the global balance closes
tau_f = t_fvm['tau_face'][:-1]
assert np.ptp(tau_f) / np.mean(tau_f) < 1e-6, np.ptp(tau_f)
assert abs(t_fvm['tau_surface'] / t_fvm['tau_face'][-2] - 1.0) < 1e-6
mbe_fvm = momentum_balance(t_fvm, z0_ground, Cd, lad_test, 0.0)
assert abs(mbe_fvm['eps_tot_rel']) < 1e-8, mbe_fvm['eps_tot_rel']

# 3) emergent roughness length equals the input
z0_fvm = emergent_z0(t_fvm['z'], t_fvm['U'])
assert abs(z0_fvm / z0_ground - 1.0) < 1e-6, z0_fvm

print(f'1) U0 error   : FVM {100*err_fvm:.3f} %, FDM {100*err_fdm:.3f} % '
      f'(exact U0 = {U0_exact:.3f} u*)')
print(f'2) tau spread : {np.ptp(tau_f):.2e}, global MBE {mbe_fvm["eps_tot_rel"]:.2e} of u*^2')
print(f'3) emergent z0: {z0_fvm:.4e} m vs input {z0_ground:.4e} m')

In [ ]:
# 4) limits of the conductance boundary condition
lim_inf = closure_1_model_u_fdm(z_test_c, z0_ground, Cd, lad_test, 0.0,
                                Utop=Utop_test, gm=1e6)
lim_zero = closure_1_model_u_fdm(z_test_c, z0_ground, Cd, lad_test, 0.0,
                                 Utop=Utop_test, gm=1e-12)
ref_dir = closure_1_model_u_fdm(z_test_c, z0_ground, Cd, lad_test, 0.0,
                                Utop=Utop_test, Ubot=0.0, lbc='dirichlet')
ref_flux = closure_1_model_u_fdm(z_test_c, z0_ground, Cd, lad_test, 0.0,
                                 Utop=Utop_test, lbc='flux')

# note: the Dirichlet reference uses ground = 'shift' and the conductance run
# ground = 'max', so the two differ by O(z0/z) in the lowest cells by design;
# the limit is therefore checked to a few permille, not to round-off
assert lim_inf['U'][0] < 1e-2 * ref_dir['U'][1]
assert abs(lim_inf['U'][1] / ref_dir['U'][1] - 1.0) < 5e-3
assert abs(lim_zero['U'][0] / ref_flux['U'][0] - 1.0) < 1e-4

print(f'4) gm -> inf : U0 = {lim_inf["U"][0]:.2e} u* (no-slip), '
      f'U1 = {lim_inf["U"][1]:.4f} vs Dirichlet {ref_dir["U"][1]:.4f}')
print(f'   gm -> 0   : U0 = {lim_zero["U"][0]:.4f} vs zero-flux {ref_flux["U"][0]:.4f}')

# 5) grid independence of the conductance + FVM solution
U_at_1m = {}
for dz_test in (0.5, 0.25, 0.1):
    zf = np.arange(0.0, z_top + 0.5 * dz_test, dz_test)
    zc = 0.5 * (zf[:-1] + zf[1:])
    r = closure_1_model_u_fvm(zf, z0_ground, Cd, np.zeros(len(zc)), 0.0,
                              Utop=np.log(z_top / z0_ground) / VON_KARMAN)
    U_at_1m[dz_test] = float(np.interp(1.0, zc, r['U']))

spread = (max(U_at_1m.values()) - min(U_at_1m.values())) / np.mean(list(U_at_1m.values()))
assert spread < 0.01, U_at_1m
print('5) U(1 m)/u* vs dz: ' + ', '.join(f'{k} m: {v:.4f}' for k, v in U_at_1m.items())
      + f'  -> spread {100*spread:.2f} %')
print('\nAll acceptance tests passed.')

# Runs

The four methods are solved with the scenario defined above.

In [ ]:
cases = [('old_orig', 'old', z_orig, lad_orig),
         ('old_fine', 'old', z_fine, lad_fine),
         ('fdm_gm', 'fdm', z_cc, lad_cc),
         ('fvm_gm', 'fvm', z_faces, lad_cc),
         ('fvm_gm_opa', 'fvm_opa', z_orig, lad_orig)]

results = {name: run_case(name, kind, grid, lad) for name, kind, grid, lad in cases}

STYLE = {'old_orig': ('tab:red', '-', 'old solver, dz = 0.25 m'),
         'old_fine': ('k', '--', f'old solver, dz = {dz_fine} m'),
         'fdm_gm': ('tab:blue', '-.', 'nodal + $g_m$'),
         'fvm_gm': ('tab:green', '-', 'FVM + $g_m$'),
         'fvm_gm_opa': ('tab:orange', ':', 'FVM + $g_m$ (OPA)')}

for name, r in results.items():
    flag = '' if r['converged'] else '   *** NOT CONVERGED ***'
    print(f"{name:9}: {r['iterations']:3d} iterations, err = {r['err']:.2e}, "
          f"{1e3*r['wall_time']:8.2f} ms{flag}")

## Wind, shear stress and eddy diffusivity

Panel b) is the diagnostic one: on a logarithmic height axis the log law is a
straight line, and any method that gets the surface layer right must follow it down
to its own lowest solution point.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
ax = axes.ravel()

z_log = np.logspace(np.log10(z0_ground), np.log10(z_top), 200)
U_log = u_star / VON_KARMAN * np.log(z_log / z0_ground)

for name, r in results.items():
    color, ls, label = STYLE[name]
    ax[0].plot(r['U_dim'], r['z'], ls, color=color, label=label)
    ax[1].plot(r['U_dim'], r['z'], ls, color=color, marker='.', ms=3, label=label)
    ax[2].plot(r['tau_dim'], r['z'], ls, color=color, label=label)
    ax[3].plot(r['Km_dim'], r['z'], ls, color=color, label=label)

for a in (ax[0], ax[1]):
    a.plot(U_log, z_log, ':', color='0.4', lw=2, label='log law')
ax[1].set_yscale('log')
ax[1].set_ylim(0.5 * z0_ground, z_top)
if LAD_KIND == 'zero':
    ax[2].axvline(u_star ** 2, color='0.4', ls=':', lw=2, label='$u_*^2$')

ax[0].set_xlabel('U [m s$^{-1}$]')
ax[1].set_xlabel('U [m s$^{-1}$]')
ax[2].set_xlabel(r'$\tau/\rho$ [m$^2$ s$^{-2}$]')
ax[3].set_xlabel('$K_m$ [m$^2$ s$^{-1}$]')
for a, letter in zip(ax, 'abcd'):
    a.set_ylabel('z [m]')
    a.text(0.02, 0.98, f'{letter})', transform=a.transAxes, va='top')
ax[0].legend(frameon=False, fontsize=9, loc='upper left', bbox_to_anchor=(0, 0.9))
fig.suptitle(f"LAD_KIND = '{LAD_KIND}'")
fig.tight_layout()

In [ ]:
z_probe = z_cc[0]     # lowest cell centre of the model grid, 0.125 m
U_probe_log = u_star / VON_KARMAN * np.log(z_probe / z0_ground)

print(f'diagnostics at z = {z_probe:.3f} m   (log law: U = {U_probe_log:.3f} m s-1, '
      f'valid as a reference only for LAD = 0)\n')
print(f"{'case':10}{'U [m/s]':>10}{'U/U_log':>10}{'u* [m/s]':>10}"
      f"{'tau_b/tau_t':>13}{'z0_emerg [m]':>14}{'d [m]':>8}")
for name, r in results.items():
    m = r['mbe']
    U0 = float(np.interp(z_probe, r['z'], r['U_dim']))
    print(f"{name:10}{U0:10.4f}{U0/U_probe_log:10.3f}"
          f"{np.sqrt(abs(m['tau_top']))*u_star:10.4f}"
          f"{m['tau_bottom']/m['tau_top']:13.4f}{r['z0_emergent']:14.3e}{r['d']:8.2f}")

### Reading the wind figure

- **`old_orig` (red).** The lowest nodes fall far below the log law: all of the
  velocity change between $z_0$ and the first node is missing, and the profile fit
  reports an emergent $z_0$ one to two orders of magnitude larger than the physical
  one. This is the wind speed the snow model currently receives.
- **`old_fine` (black).** The same scheme with the log layer resolved recovers the
  log law and the correct emergent $z_0$ — the physics of the closure is fine, the
  resolution was not. The price is the wall time printed above.
- **`fdm_gm` (blue) and `fvm_gm` (green).** The conductance boundary condition puts
  the lowest cell back on the log law at the original resolution. The two differ only
  in the discretization, and the difference between them is exactly the
  non-conservation of the nodal scheme, visible in panel c) as a shear stress that
  drifts with height instead of staying constant.

## Scalar profiles

Same four methods, same sources. The markers at $z = z_0$ are the surface values:
for the conductance runs they come from the analytic jump $F/(\rho_{mol} g_s U_0)$,
for the nodal runs they are simply the value at the ground node.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
panels = [('CO2', 'CO$_2$ [ppm]', 'CO2_surf'),
          ('H2O', 'H$_2$O [mmol mol$^{-1}$]', 'H2O_surf'),
          ('T', 'T [$^\\circ$C]', 'T_surf')]

for a, (key, xlabel, surf_key) in zip(axes, panels):
    for name, r in results.items():
        color, ls, label = STYLE[name]
        a.plot(r[key], r['z'], ls, color=color, label=label)
        a.plot(r[surf_key], z0_ground, 'o', color=color, ms=6, mfc='none')
    a.set_yscale('log')
    a.set_ylim(0.5 * z0_ground, z_top)
    a.set_xlabel(xlabel)
    a.set_ylabel('z [m]')

    # keep one diverging solution from destroying the axis
    vals = np.concatenate([np.atleast_1d(results[n][key]) for n in results
                           if results[n]['converged']])
    lo, hi = np.nanmin(vals), np.nanmax(vals)
    pad = 0.15 * (hi - lo + EPS)
    a.set_xlim(lo - pad, hi + pad)

for a, letter in zip(axes, 'abc'):
    a.text(0.02, 0.98, f'{letter})', transform=a.transAxes, va='top')
axes[0].legend(frameon=False, fontsize=9)
fig.suptitle(f"LAD_KIND = '{LAD_KIND}', open circles = surface value at $z_0$")
fig.tight_layout()

In [ ]:
rows = []
for name, r in results.items():
    has_gs = np.isfinite(r['gs'])
    rows.append({'case': name,
                 'U at lowest point [m/s]': r['U_dim'][0],
                 'z lowest point [m]': r['z'][0],
                 'g_s [-]': r['gs'],
                 'g_s*U0 [m/s]': r['gs'] * r['U_dim'][0] if has_gs else np.nan,
                 'CO2 surface [ppm]': r['CO2_surf'],
                 'dCO2 surface-air [ppm]': (r['CO2_surf'] - r['CO2'][0]) if has_gs else np.nan,
                 'T surface [degC]': r['T_surf'],
                 'dT surface-air [K]': (r['T_surf'] - r['T'][0]) if has_gs else np.nan,
                 'converged': r['converged']})
surface_table = pd.DataFrame(rows).set_index('case')
surface_table.round(4)

### Reading the scalar figure

The scalar profiles inherit the wind error twice over: through $K_s = Sc\,K_m$ and
through the surface resistance. With the nodal formulation the whole resistance of
the unresolved layer has to be carried by $K_{1/2}$, which the coarse grid gets
wrong, so the surface concentration and temperature jumps are misrepresented — the
`old_orig` surface values are the ones a snow or forest-floor scheme would be handed
today. The conductance runs report the aerodynamic conductance $g_s U_0$ explicitly
(the table above), which is exactly the quantity those schemes need; note that it is
built with $g_s = g_m Sc$, not $g_m/Sc$.

## Momentum balance

Panel a) shows where in the column each scheme creates or destroys momentum,
panel b) the global residual, and panel c) how much of the momentum flux actually
reaches the ground.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
ax = axes.ravel()
names = list(results.keys())

# a) reconstructed momentum flux through every face: a constant-flux layer is a
#    vertical line at 1 on open ground
for name, r in results.items():
    color, ls, label = STYLE[name]
    m = r['mbe']
    ax[0].plot(m['tau_face'] / m['tau_top'], m['z_face'], ls, color=color, label=label)
    ax[0].plot(m['tau_bottom'] / m['tau_top'], max(m['z_face'][0] * 0.5, z0_ground),
               'o', color=color, ms=6, mfc='none')
ax[0].axvline(1.0, color='0.4', ls=':', lw=2)
ax[0].set_yscale('log')
ax[0].set_ylim(0.5 * z0_ground, z_top)
ax[0].set_xlim(-0.1, 1.6)
ax[0].set_xlabel(r'$\tau_{i+1/2}/\tau_{top}$ [-]')
ax[0].legend(frameon=False, fontsize=9)

# b) where in the column the imbalance accumulates
for name, r in results.items():
    color, ls, label = STYLE[name]
    m = r['mbe']
    ax[1].plot(100 * np.cumsum(m['eps_rel']), m['z'], ls, color=color)
ax[1].axvline(0.0, color='0.4', ls=':', lw=2)
ax[1].set_yscale('log')
ax[1].set_ylim(0.5 * dz, z_top)
ax[1].set_xlabel(r'cumulative $\sum\varepsilon_i / u_*^2$ [%]')

# c) global residual
glob = [100 * abs(results[n]['mbe']['eps_tot_rel']) for n in names]
ax[2].bar(range(len(names)), np.maximum(glob, 1e-12),
          color=[STYLE[n][0] for n in names])
ax[2].set_yscale('log')
ax[2].set_xticks(range(len(names)))
ax[2].set_xticklabels(names, rotation=20)
ax[2].set_ylabel(r'$|\varepsilon_{tot}| / u_*^2$ [%]')

# d) how much of the stress actually reaches the surface
ratio = [results[n]['mbe']['tau_bottom'] / results[n]['mbe']['tau_top'] for n in names]
ax[3].bar(range(len(names)), ratio, color=[STYLE[n][0] for n in names])
ax[3].axhline(1.0, color='0.4', ls=':', lw=2)
ax[3].set_xticks(range(len(names)))
ax[3].set_xticklabels(names, rotation=20)
ax[3].set_ylabel(r'$\tau_{bottom}/\tau_{top}$ [-]')

for a in (ax[0], ax[1]):
    a.set_ylabel('z [m]')
for a, letter in zip(ax, 'abcd'):
    a.text(0.02, 0.98, f'{letter})', transform=a.transAxes, va='top')
fig.suptitle(f"Momentum balance, LAD_KIND = '{LAD_KIND}' "
             '(dotted lines: exact constant-flux layer)')
fig.tight_layout()

### Reading the momentum balance figure

$\varepsilon_i$ measures how badly cell $i$ fails its own flux budget. The nodal
schemes concentrate their error in the lowest few cells, where $K_m$ curves most
sharply and the expanded form $K_mU'' + K_m'U'$ has no shared face flux between
neighbouring equations; refining the grid does not remove it, because on a
log profile the relative curvature of the lowest cells is the same at every $\Delta z$.
The FVM scheme is conservative by construction: its residual is at round-off, and
panel c) shows $\tau_{bottom}/\tau_{top} = 1$ for the open-ground case, i.e. the
momentum extracted at the surface equals the momentum supplied at the top. For the
nodal runs $\tau_{bottom}$ is the flux through the lowest *interior* face, so panel
c) also quantifies how much stress is lost before reaching the ground.

In [ ]:
summary = []
for name, r in results.items():
    m = r['mbe']
    summary.append({
        'case': name,
        'U(0.125 m) [m/s]': float(np.interp(z_probe, r['z'], r['U_dim'])),
        'u* [m/s]': float(np.sqrt(abs(m['tau_top'])) * u_star),
        'tau_bottom/tau_top': m['tau_bottom'] / m['tau_top'],
        'z0 emergent [m]': r['z0_emergent'],
        'global MBE [% of u*^2]': 100 * m['eps_tot_rel'],
        'g_m [-]': r['gm'],
        'iterations': r['iterations'],
        'wall time [ms]': 1e3 * r['wall_time'],
        'converged': r['converged']})
summary_table = pd.DataFrame(summary).set_index('case')
summary_table.round(5)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

# Absolute difference between FVM and FVM-OPA models
r_fvm = results['fvm_gm']
r_opa = results['fvm_gm_opa']

U_fvm = r_fvm['U_dim']
U_opa = r_opa['U_dim']
z = r_fvm['z']

# Absolute difference
abs_diff = np.abs(U_fvm - U_opa)

ax.loglog(abs_diff, z, '-', color='tab:purple', linewidth=2, label='$|U_{FVM} - U_{OPA}|$')
ax.set_yscale('log')
ax.set_ylim(0.5 * z0_ground, z_top)
ax.set_xlabel('Absolute difference [m s$^{-1}$]')
ax.set_ylabel('z [m]')
ax.set_title('Absolute difference between FVM and FVM-OPA models')
ax.legend(frameon=False, fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()

# Conclusions

1. **The mixing length fix alone is not enough.** With $\ell = \kappa(z+z_0)$ but a
   Dirichlet no-slip at the lowest node, a $\Delta z = 0.25$ m grid still misses the
   whole log layer; only refining to $\Delta z \sim z_0$ recovers the log law, at a
   cost of two to three orders of magnitude in wall time.
2. **The surface conductance recovers it at the original resolution.** $g_m$ is one
   line of algebra, it enters the lowest cell exactly like the canopy drag term, and
   the existing no-slip and zero-flux branches are its two limits — so nothing that
   works today has to break.
3. **The conservative discretization is the other half.** The conductance BC fixes
   the boundary; only the flux-form (FVM) discretization also makes the interior
   conserve momentum. On open ground it reproduces the log law and the input $z_0$
   essentially exactly, at any resolution.
4. **What to change in pyAPES** (`pyAPES/microclimate/micromet.py`):
   - `mixing_length`: remove the grid-dependent constants, use `np.maximum` with
     `l_min = kappa*z0` and fall back to the open-ground branch when the canopy is
     unresolved;
   - `closure_1_model_U`: `dz = z[1] - z[0]`, identity tests for the boundary
     switches, and the conductance lower boundary condition;
   - `Micromet.update_state`: the patches `U[0] = U[1]` and `Km[0] = Km[1]` exist
     only because the lowest node is meaningless today; with the conductance BC they
     should be dropped;
   - `Micromet.scalar_profiles`: pass $g_s = g_m Sc$ and the surface value from the
     analytic jump to the snow and forest-floor schemes instead of letting them read
     the lowest node.
5. **Caveat for unresolved canopies.** When `hc_eff < 3*dz` the coarse grid cannot
   represent the canopy at all and `mixing_length` falls back to the open-ground
   branch; the conductance then uses the bare-ground $z_0$. A short canopy should be
   handed to the solver as an effective $z_0$ and displacement height, not as a
   plant area density profile — the `'constant'` scenario with a small `h_const`
   shows what happens if it is not.

In [ ]:
np.linspace(0,25,101)

In [ ]:
np.linspace(0.125, 24.875, 100)

In [ ]:
# --- the deliverable for pyAPES: profiles on the original grid, 0 ... 25 m, dz = 0.25 m
r = results['fvm_gm_opa']
print(f"pyAPES grid: {len(r['z_pyapes'])} nodes, "
      f"{r['z_pyapes'][0]:.2f} ... {r['z_pyapes'][-1]:.2f} m, "
      f"dz = {r['z_pyapes'][1] - r['z_pyapes'][0]:.3f} m")
print(f"{'z [m]':>8}{'U [m/s]':>10}{'Km [m2/s]':>12}{'tau [m2/s2]':>14}")
for i in list(range(6)) + [12, 20, 40, 60, 80, 99, 100]:
    print(f"{r['z_pyapes'][i]:8.2f}{r['U_pyapes'][i]:10.4f}"
          f"{r['Km_pyapes'][i]:12.5f}{r['tau_pyapes'][i]:14.6f}")

# the surface exchange that replaces the meaningless U[0]/Km[0] of the present
# pyAPES code: tau_0 = g_m*U_0^2, and g_m*U_0 is the aerodynamic conductance
print(f"\ng_m = {r['gm']:.5f}, U(lowest cell centre) = {r['U_dim'][0]:.4f} m/s, "
      f"g_m*U0 = {r['gm'] * r['U_dim'][0]:.5f} m/s, "
      f"tau_surface = {r['tau_pyapes'][0]:.6f} m2 s-2")
